In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import uuid

# ==========================================================
# Configuration
# ==========================================================
bronze_table = "bronze_dev.global_mart_retail.raw_data"
silver_table_dim_customer = "silver_dev.global_mart_retail.dim_customer"

# ==========================================================
# Generate ONE batch_id for entire run
# ==========================================================
BATCH_ID = str(uuid.uuid4())

# ==========================================================
# 1. Read Bronze Data
# ==========================================================
bronze_df = spark.read.table(bronze_table)

# ==========================================================
# 2. Clean & Standardize Customer Attributes
# ==========================================================
cleaned_df = (
    bronze_df
    .select(
        F.upper(F.trim(F.col("customer_id"))).alias("customer_id"),
        F.lower(F.trim(F.col("customer_name"))).alias("customer_name"),
        F.coalesce(F.lower(F.trim(F.col("segment"))), F.lit("unknown")).alias("customer_segment"),
        F.coalesce(F.lower(F.trim(F.col("country"))), F.lit("unknown")).alias("country"),
        F.coalesce(F.lower(F.trim(F.col("city"))), F.lit("unknown")).alias("city"),
        F.coalesce(F.lower(F.trim(F.col("state"))), F.lit("unknown")).alias("state"),
        F.lpad(
            F.regexp_replace(F.col("postal_code").cast("string"), "[^0-9]", ""),
            5,
            "0"
        ).alias("postal_code"),
        F.coalesce(F.lower(F.trim(F.col("region"))), F.lit("unknown")).alias("region"),
        F.col("ingestion_ts"),
        F.col("row_id")
    )
)

# ==========================================================
# 3. Generate Business Hash
# ==========================================================
hashed_df = (
    cleaned_df
    .withColumn(
        "customer_hash",
        F.sha2(
            F.concat_ws(
                "||",
                "customer_name",
                "customer_segment",
                "country",
                "city",
                "state",
                "postal_code",
                "region"
            ),
            256
        )
    )
)

# ==========================================================
# 4. HARD SOURCE DE-DUPLICATION
# ==========================================================
dedup_window = (
    Window
    .partitionBy("customer_id", "customer_hash")
    .orderBy(F.col("ingestion_ts"), F.col("row_id").desc())
)

deduped_df = (
    hashed_df
    .withColumn("rn", F.row_number().over(dedup_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# ==========================================================
# 5. Assign is_current within THIS batch
# ==========================================================
current_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("ingestion_ts"), F.col("row_id").desc())
)

staged_df = (
    deduped_df
    .withColumn("rn", F.row_number().over(current_window))
    .withColumn("is_current_record", F.col("rn") == 1)
    .withColumn("effective_start_timestamp", F.col("ingestion_ts"))
    .withColumn("effective_end_timestamp", F.lit(None).cast("timestamp"))
    .withColumn("load_timestamp", F.current_timestamp())
    .withColumn("batch_id", F.lit(BATCH_ID))
    .drop("rn")
)

# ==========================================================
# 6. Create Silver Table if Not Exists
# ==========================================================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_table_dim_customer} (
    customer_key BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id STRING,
    customer_name STRING,
    customer_segment STRING,
    country STRING,
    city STRING,
    state STRING,
    postal_code STRING,
    region STRING,
    customer_hash STRING,
    effective_start_timestamp TIMESTAMP,
    effective_end_timestamp TIMESTAMP,
    is_current_record BOOLEAN,
    load_timestamp TIMESTAMP,
    batch_id STRING
)
USING DELTA
""")

silver_delta = DeltaTable.forName(spark, silver_table_dim_customer)

# ==========================================================
# 7. MERGE — INSERT NEW HASHES ONLY
# ==========================================================
(
    silver_delta.alias("t")
    .merge(
        staged_df.alias("s"),
        "t.customer_id = s.customer_id AND t.customer_hash = s.customer_hash"
    )
    .whenNotMatchedInsert(
        values={
            "customer_id": "s.customer_id",
            "customer_name": "s.customer_name",
            "customer_segment": "s.customer_segment",
            "country": "s.country",
            "city": "s.city",
            "state": "s.state",
            "postal_code": "s.postal_code",
            "region": "s.region",
            "customer_hash": "s.customer_hash",
            "effective_start_timestamp": "s.effective_start_timestamp",
            "effective_end_timestamp": "s.effective_end_timestamp",
            "is_current_record": "s.is_current_record",
            "load_timestamp": "s.load_timestamp",
            "batch_id": "s.batch_id"
        }
    )
    .execute()
)

# ==========================================================
# 8. FIX is_current & effective_end_timestamp (SCD2)
# ==========================================================
silver_df = spark.read.table(silver_table_dim_customer)

scd_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("effective_start_timestamp").desc())
)

scd_updates = (
    silver_df
    .withColumn("rn", F.row_number().over(scd_window))
    .withColumn(
        "new_is_current",
        F.col("rn") == 1
    )
    .withColumn(
        "new_effective_end_timestamp",
        F.when(
            F.col("rn") == 1,
            None
        ).otherwise(
            F.lag("effective_start_timestamp").over(scd_window)
        )
    )
    .select(
        "customer_key",
        "new_is_current",
        "new_effective_end_timestamp"
    )
)

(
    silver_delta.alias("t")
    .merge(
        scd_updates.alias("s"),
        "t.customer_key = s.customer_key"
    )
    .whenMatchedUpdate(
        set={
            "is_current_record": "s.new_is_current",
            "effective_end_timestamp": "s.new_effective_end_timestamp"
        }
    )
    .execute()
)

print(f"✅ SCD Type 2 load completed successfully | batch_id = {BATCH_ID}")


In [0]:

# ==========================================================
# 6. Reconciliation & Validation Checks
# ==========================================================
spark.sql(f"""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT customer_id) AS distinct_customers,
    SUM(CASE WHEN is_current_record THEN 1 ELSE 0 END) AS current_records
FROM {silver_table_dim_customer}
""").show()

In [0]:
spark.sql(f"""
SELECT customer_id, COUNT(*) AS versions
FROM {silver_table_dim_customer}
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY versions DESC
""").show()

In [0]:
%sql
select * from silver_dev.global_mart_retail.dim_customer where customer_id = 'SV-20365';